# Publish Model to HF Hub

Select a model checkpoint from W&B artifacts and publish it as the official
`kaya-go/moku-v2` model on Hugging Face Hub.

**Workflow:**

1. List available W&B model artifacts (with mAP, epoch, run info)
2. Set the artifact path to publish
3. Download and load the model
4. Quick sanity check on real test set
5. Push to HF Hub

## Available Artifacts

Browse model artifacts from W&B and pick the one to publish.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv

load_dotenv()

from moku.dataset import CATEGORIES, ID_TO_CATEGORY
from moku.evaluation import evaluate_map, format_map_per_class, format_map_results
from moku.runs import list_wandb_model_artifacts, load_model_from_wandb
from moku.model import make_eval_transform

## Select Artifact

Set the artifact path below. Use the `name` column from the table above.
Append `:latest` or `:vN` for a specific version.

In [ ]:
# ← Set the artifact to publish
ARTIFACT = "model-r4_lr5e-4_cos200:latest"

HF_MODEL = "kaya-go/moku-v2"
HF_DATASET = "kaya-go/moku-v2"

print(f"Will publish: {ARTIFACT}")
print(f"Target: {HF_MODEL}")

## Download & Load Model

In [ ]:
ip, model = load_model_from_wandb(ARTIFACT)

total = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total:,} parameters")
print(f"Labels: {model.config.id2label}")

## Sanity Check — mAP on Real Test

Quick evaluation on the real test split to confirm the model performs as expected.

In [ ]:
ds_real = load_dataset(HF_DATASET, "real")
ds_real.set_transform(make_eval_transform(ip))

metrics = evaluate_map(
    model=model,
    dataset=ds_real["test"],
    image_processor=ip,
    batch_size=8,
)

print("=== Overall mAP ===")
display(format_map_results(metrics))

print("\n=== Per-Class AP ===")
display(format_map_per_class(metrics))

## Push to HF Hub

Publish the model and image processor to the official HF model repository.

**Warning**: This overwrites the current model on the `main` branch of `kaya-go/moku-v2`.

In [ ]:
# Uncomment to push (destructive — overwrites current model on HF Hub)
# model.push_to_hub(HF_MODEL)
# ip.push_to_hub(HF_MODEL)
# print(f"Model pushed to https://huggingface.co/{HF_MODEL}")